# Ekans Tutorial: Dynamic, Flattened Module Access

## 0. Introduction

Tuning quantum chips is inherently iterative: you run measurements, analyse the results, tweak parameters, and repeat. In practice, this also means frequently modifying your measurement scripts, configurations, and helper functions.

However, Python’s import system is static. Once a module is imported, the running interpreter does not automatically pick up changes you make on disk. This forces you to either restart your session or manually reload modules—both of which interrupt workflow and can become error-prone in complex projects.

This is exactly the problem that Ekans solves.

## 1. What is Ekans?

Ekans is a lightweight proxy layer that sits on top of your package and provides:

**A live view of your codebase**

>It mirrors your folder and module structure, so your package is accessible exactly as it exists on disk.

**Clean, nested access via namespaces**

>Modules and submodules can be accessed in an intuitive, hierarchical way—no need for repetitive or verbose imports.

**One-line dynamic reloading**

>Changes to any part of your codebase can be picked up instantly with a single reload call with no kernel restarts required.

**With Ekans, you can:**

- 🔁 Iterate faster — modify code and immediately test it without restarting your environment

- 🧠 Reduce mental overhead — no need to track what needs re-importing
- 🧩 Work with large codebases comfortably — even deeply nested module structures stay manageable
- ⚡ Stay focused on experiments — less time fighting Python imports, more time analysing results

This notebook demonstrates how to use Ekans to dynamically load, reload, and interact with a structured Python package in a seamless way.

## 2. Hands on example

We will use the arbok_driver.examples package. This might look intimidating now but will be a handy reference once you start playing around yourself.

Let's start how  one would normally set up a driver and a generic measurement.

In [6]:
from arbok_driver import ArbokDriver, Device, Ekans, Measurement

In [ ]:
from arbok_driver.examples.sub_sequences import SquarePulseScalable
from arbok_driver.examples.configurations import (
    opx1000_config,
    divider_config,
    square_pulse_scalable_conf
)

In [3]:
mock_device = Device(
    name = 'mock_device',
    opx_config = opx1000_config,
    divider_config = divider_config)

qm_driver = ArbokDriver('qm_driver', mock_device)

In [4]:
mock_measurement = Measurement(qm_driver, 'mock_measurement')

In [5]:
square_pulse_scaled = SquarePulseScalable(
    mock_measurement,
    'square_pulse_scaled',
    square_pulse_scalable_conf
    )

Here we used direct imports from our modules but we can make that dynamic using `Ekans`. Insead of importing the files directly as shown above, you can create an `Ekans` instance for a given (sub)module.

In [7]:
from arbok_driver import examples
ek_sequences = Ekans(examples.sub_sequences)
ek_configs = Ekans(examples.configurations)

Ekans now replicates your folder structure and exposes all objects available in leaf modules automatically. If you manually expose an object via an `__init__.py` file, it will be avaialbe wherever you want it as well.

Creating an `Ekans` instance will:

- Discover all submodules
- Import or reload them
- Build a dynamic attribute tree
- Flatten submodule attributes

In [8]:
ek_configs.divider_config

{'P1': {'division': 6},
 'P2': {'division': 6},
 'P3': {'division': 6},
 'gate_2': {'division': 2},
 'readout_element': {'division': 2}}

In [14]:
ek_configs.hardware_configs.opx1000_config

<module 'arbok_driver.examples.configurations.hardware_configs.opx1000_config' from '/home/nixos/git-repos/arbok_driver/arbok_driver/examples/configurations/hardware_configs/opx1000_config.py'>

In [15]:
ek_sequences.SquarePulse

arbok_driver.examples.sub_sequences.square_pulse.SquarePulse

Have a look at the given folder structure here and play around!

```
arbok_driver
├── ...
├── examples
│   ├── configurations
│   │   ├── __init__.py
│   │   ├── hardware_configs
│   │   │   ├── divider_config.py
│   │   │   ├── __init__.py
│   │   │   ├── opx1000_config.py
│   │   └── sequence_configs
│   │       ├── __init__.py
│   │       ├── coulomb_peaks_config.py
│   │       ├── device_config.py
│   │       ├── parity_init_conf.py
│   │       ├── parity_readout_conf.py
│   │       ├── square_pulse_confs.py
│   │       └── stability_map_config.py
│   ├── experiments
│   │   ├── __init__.py
│   │   └── square_pulse_experiments.py
│   ├── readout_classes
│   │   ├── __init__.py
│   │   ├── dc_average.py
│   │   ├── dc_chopped_readout.py
│   │   ├── difference.py
│   │   └── threshold.py
│   └── sub_sequences
│       ├── __init__.py
│       ├── coulomb_peaks.py
│       ├── parity_initialization.py
│       ├── parity_readout.py
│       ├── square_pulse_legacy.py
│       ├── square_pulse.py
│       ├── square_pulse_scalable.py
│       └── stability_map.py
├── ...
```

Each submodule contains Python files defining classes/functions.

## 3. Live module re-loading 

Assume one has made changes to files on disc. Use `reload_modules` to update the changes from disk and make them available to you.

In [16]:
ek_configs.reload_modules()

In [17]:
ek_sequences.reload_modules()

This will:

- reload all modules
- rebuild the attribute tree
- reflect code changes without restarting Python

Now the previous measurement can be re-constructed and run.

In [20]:
qm_driver.reset_measurements()

mock_measurement = Measurement(qm_driver, 'mock_measurement')

refreshed_sequence = ek_sequences.SquarePulseScalable(
    mock_measurement,
    'square_pulse_scaled',
    ek_configs.square_pulse_scalable_conf
)

In [ ]:
square_pulse_scaled = SquarePulseScalable(
    mock_measurement,
    'square_pulse_scaled',
    square_pulse_scalable_conf
    )

## 4. Rules and limitations

To ensure `Ekans` works reliably, your package should follow these conventions:

### 1. Every folder must be a Python package
Each directory **must contain an `__init__.py` file**.

- This ensures Python treats the directory as a proper package  
- Without it, Python may create a *namespace package*, which can break module discovery and reloading in `Ekans`

---

### 2. Only `.py` files are treated as modules
`Ekans` only considers Python source files:

- ✅ `.py` files are loaded and exposed as modules  
- 🚫 Other file types (e.g. `.txt`, `.json`, `.yaml`) are ignored  

These non-Python files do **not** interfere with `Ekans`, but they will not be accessible through it.

---

### 3. Modules must be importable
All Python modules should be valid at import time:

- They must not raise exceptions during import (unless explicitly intended)  
- All internal and external imports must resolve correctly  

If a module fails to import, it may prevent parts of your package from being available through `Ekans`.

---

Following these rules ensures that `Ekans` can correctly discover, structure, and dynamically reload your codebase.

## 5. Summary

Traditional Python imports are static, meaning that changes to your codebase require manual reloading or restarting your session. This becomes cumbersome when working with complex, modular projects that evolve rapidly during development and experimentation.

`Ekans` addresses this by providing a dynamic interface to your package:

- It mirrors your folder structure as a nested, intuitive namespace  
- It allows seamless access to modules and submodules without repetitive imports  
- It enables **one-line reloading**, so changes on disk are immediately reflected in your running session  

By following a few simple rules—ensuring all directories are proper packages, keeping modules importable, and structuring your code cleanly—you can fully leverage `Ekans` to create a more fluid and efficient development workflow.

### Key takeaway

> `Ekans` lets you treat your codebase as a **live, evolving object**, rather than a static snapshot.

This reduces friction, speeds up iteration, and helps you stay focused on what matters most: designing experiments and analysing results.